Social Media-Aware Cleaning 

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import emoji
import re

In [2]:
spark = SparkSession.builder.getOrCreate()

26/03/26 16:01:32 WARN Utils: Your hostname, Meenas-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.127 instead (on interface en0)
26/03/26 16:01:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 16:01:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
train_df = spark.read.csv('data/twitter_training.csv')

In [4]:
col_names = ["id", "topic", "sentiment", "content"]
train_df = train_df.toDF(*col_names)

train_df.show(5, truncate=False)

+----+-----------+---------+---------------------------------------------------------+
|id  |topic      |sentiment|content                                                  |
+----+-----------+---------+---------------------------------------------------------+
|2401|Borderlands|Positive |im getting on borderlands and i will murder you all ,    |
|2401|Borderlands|Positive |I am coming to the borders and I will kill you all,      |
|2401|Borderlands|Positive |im getting on borderlands and i will kill you all,       |
|2401|Borderlands|Positive |im coming on borderlands and i will murder you all,      |
|2401|Borderlands|Positive |im getting on borderlands 2 and i will murder you me all,|
+----+-----------+---------+---------------------------------------------------------+
only showing top 5 rows



(1) Emoji Translation

In [5]:
def translate_emojis(text):
    if text is None:
        return None
    
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    text = text.replace(":", "")
    
    return text

In [6]:
emoji_udf = udf(translate_emojis, StringType())

In [7]:
emoji_df = train_df.withColumn("emoji_translated", emoji_udf(train_df["content"]))
emoji_df.select("content", "emoji_translated").show(5, truncate=False)

+---------------------------------------------------------+---------------------------------------------------------+
|content                                                  |emoji_translated                                         |
+---------------------------------------------------------+---------------------------------------------------------+
|im getting on borderlands and i will murder you all ,    |im getting on borderlands and i will murder you all ,    |
|I am coming to the borders and I will kill you all,      |I am coming to the borders and I will kill you all,      |
|im getting on borderlands and i will kill you all,       |im getting on borderlands and i will kill you all,       |
|im coming on borderlands and i will murder you all,      |im coming on borderlands and i will murder you all,      |
|im getting on borderlands 2 and i will murder you me all,|im getting on borderlands 2 and i will murder you me all,|
+-------------------------------------------------------

In [8]:
# JUST SOME TEST DATA TO SEE IF THE EMOJI TRANSLATION METHOD IS WORKING lol

emoji_test_data = [
    ("1", "test", "Positive", "I love this😂"),
    ("2", "test", "Negative", "This is terrible 😭 "),
    ("3", "test", "Neutral", "Nothing special 😐"),
    ("4", "test", "Positive", "Best day ever :)"),
    ("5", "test", "Negative", "I'm tired😩 ")
]

emoji_test_df = spark.createDataFrame(
    emoji_test_data,
    ["id", "topic", "sentiment", "content"]
)

In [9]:
emoji_test_result_df = emoji_test_df.withColumn(
    "emoji_translated",
    emoji_udf(emoji_test_df["content"])
)

emoji_test_result_df.select("content", "emoji_translated").show(truncate=False)

+--------------------+--------------------------------------+
|content             |emoji_translated                      |
+--------------------+--------------------------------------+
|I love this😂       |I love this face with tears of joy    |
|This is terrible 😭 |This is terrible  loudly crying face  |
|Nothing special 😐  |Nothing special  neutral face         |
|Best day ever :)    |Best day ever )                       |
|I'm tired😩         |I'm tired weary face                  |
+--------------------+--------------------------------------+



(2) Hashtag Splitting

In [10]:
def split_hashtags(text):
    if text is None:
        return None

    def split_tag(match):
        tag = match.group()[1:]
        tag = tag.replace("_", " ")
        tag = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        return tag

    return re.sub(r'#\w+', split_tag, text)

In [11]:
hashtag_udf = udf(split_hashtags, StringType())

In [12]:
hashtag_df = train_df.withColumn("hashtags_split", hashtag_udf(train_df["content"]))
hashtag_df.select("content", "hashtags_split").show(20, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content                                                                                                                                                                                                                                                                                              |hashtags_split                                                                                                     

In [13]:
# Test data for hashtag splitting
hashtag_test_data = [
    ("1", "test", "Positive", "I love this #BestDayEver"),
    ("2", "test", "Negative", "This is terrible #WorstDay"),
    ("3", "test", "Neutral", "Just another day #MondayMood"),
    ("4", "test", "Positive", "Best day ever #BestDayEver"),
    ("5", "test", "Negative", "I hate this #WorstDay"),
    ("6", "test", "Neutral", "chilling #good_vibes"),
    ("7", "test", "Positive", "So excited #LifeIsGood"),
    ("8", "test", "Negative", "why is this happening #badluck")
]

hashtag_test_df = spark.createDataFrame(
    hashtag_test_data,
    ["id", "topic", "sentiment", "content"]
)

In [14]:
hashtag_test_result_df = hashtag_test_df.withColumn(
    "hashtags_split",
    hashtag_udf(hashtag_test_df["content"])
)

hashtag_test_result_df.select("content", "hashtags_split").show(truncate=False)

+------------------------------+-----------------------------+
|content                       |hashtags_split               |
+------------------------------+-----------------------------+
|I love this #BestDayEver      |I love this Best Day Ever    |
|This is terrible #WorstDay    |This is terrible Worst Day   |
|Just another day #MondayMood  |Just another day Monday Mood |
|Best day ever #BestDayEver    |Best day ever Best Day Ever  |
|I hate this #WorstDay         |I hate this Worst Day        |
|chilling #good_vibes          |chilling good vibes          |
|So excited #LifeIsGood        |So excited Life Is Good      |
|why is this happening #badluck|why is this happening badluck|
+------------------------------+-----------------------------+

